# CSV vs. Parquet con PySpark

Genera un millón de pedidos ficticios, guarda los mismos datos en CSV y Parquet y compara cuánto ocupan, qué tipos recupera Spark y qué resultado da la misma consulta. Antes de empezar, completa la [guía de entorno de PySpark](../01_environment_pyspark.md).

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[2]").appName("CSV vs Parquet").getOrCreate()
carpeta = Path(mkdtemp(prefix="pyspark_pedidos_"))
print(f"Archivos de la práctica: {carpeta}")

## 1. Generar los pedidos

Cada fila tiene un identificador, una zona y una cantidad de unidades.

In [ ]:
pedidos = spark.range(1_000_000, numPartitions=4).select(
    F.col("id").alias("pedido"),
    F.concat(F.lit("zona_"), F.col("id") % 20).alias("zona"),
    (F.col("id") % 12 + 1).alias("unidades"),
)
pedidos.show(5)

## 2. Guardar y comparar tamaños

Spark crea una carpeta por formato. Suma solo los archivos de datos `part-*` de cada carpeta. El tamaño refleja cómo se guardaron estos pedidos, incluida la compresión de Parquet.

In [ ]:
ruta_csv = carpeta / "pedidos_csv"
ruta_parquet = carpeta / "pedidos_parquet"

pedidos.write.csv(str(ruta_csv), header=True)
pedidos.write.parquet(str(ruta_parquet), compression="snappy")

bytes_csv = sum(archivo.stat().st_size for archivo in ruta_csv.glob("part-*.csv"))
bytes_parquet = sum(archivo.stat().st_size for archivo in ruta_parquet.glob("part-*.parquet"))

print(f"CSV:     {bytes_csv / 1024**2:.2f} MiB")
print(f"Parquet: {bytes_parquet / 1024**2:.2f} MiB")
print(f"Parquet ocupa el {100 * bytes_parquet / bytes_csv:.1f} % del tamaño del CSV.")

## 3. Volver a leer y comparar esquemas

CSV guarda texto. Usamos `inferSchema=True` para que Spark deduzca los tipos a partir de los valores. Parquet almacena los tipos junto con los datos. Compara cómo aparece `unidades` en cada esquema.

In [ ]:
pedidos_csv = spark.read.csv(str(ruta_csv), header=True, inferSchema=True)
pedidos_parquet = spark.read.parquet(str(ruta_parquet))

print("CSV")
pedidos_csv.printSchema()
print("Parquet")
pedidos_parquet.printSchema()

## 4. Sumar las unidades de `zona_3`

Aplica el mismo filtro y la misma suma a los dos DataFrames. Ambos deben dar **400 000 unidades**.

In [ ]:
total_csv = pedidos_csv.filter(F.col("zona") == "zona_3").agg(F.sum("unidades")).first()[0]
total_parquet = pedidos_parquet.filter(F.col("zona") == "zona_3").agg(F.sum("unidades")).first()[0]

print(f"CSV:     {total_csv} unidades")
print(f"Parquet: {total_parquet} unidades")
assert total_csv == total_parquet == 400_000

Si vas a continuar con el cuaderno 06, copia la ruta de Parquet que aparece a continuación.

In [ ]:
print(ruta_parquet)
spark.stop()

## Navegación

[Anterior · Ejercicios de DataFrames](04_df_exercise.ipynb) · [Volver al README](../README.md) · [Siguiente · Parquet particionado por zona](06_parquet_python_vs_pyspark.ipynb)